In [3]:
"""
gap11_clean.py
==============
Clean version of Gap 11.

For each algebra (E and G) and each length L in {2,4,6,8}:
  - Count loops under left-associative product
  - Count loops under right-associative product
  - Report sum and average of scalar values

A "loop" is a sequence (a_1, ..., a_L) of basis elements whose product
returns to a scalar multiple of identity.
"""

import numpy as np
from itertools import product

MOD = 9
N = 9
FANO = [(0,1,2), (0,3,4), (0,5,6), (1,3,5), (1,4,6), (2,3,6), (2,4,5)]


def build(t_sq, t_left, t_right):
    M = np.zeros((N, N, N), dtype=int)
    for i in range(N):
        M[7, i, i] = 1
        M[i, 7, i] = 1
    for (a, b, c) in FANO:
        M[a, b, c] = 1
        M[b, c, a] = 1
        M[c, a, b] = 1
        M[b, a, c] = -1 % MOD
        M[c, b, a] = -1 % MOD
        M[a, c, b] = -1 % MOD
    for i in range(7):
        M[i, i, 7] = -1 % MOD
    M[8, 8, 7] = t_sq % MOD
    for i in range(7):
        M[8, i, i] = t_left % MOD
        M[i, 8, i] = t_right % MOD
    M[8, 7, 7] = t_left % MOD
    M[7, 8, 8] = 1
    return M


def left_matrices(M):
    """L[i] = 9x9 matrix such that (e_i * x) = L[i] @ x."""
    L = []
    for i in range(N):
        Lmat = np.zeros((N, N), dtype=int)
        for j in range(N):
            for k in range(N):
                Lmat[k, j] = M[i, j, k] % MOD
        L.append(Lmat)
    return L


def right_matrices(M):
    """R[i] = 9x9 matrix such that (x * e_i) = R[i] @ x."""
    R = []
    for i in range(N):
        Rmat = np.zeros((N, N), dtype=int)
        for j in range(N):
            for k in range(N):
                Rmat[k, j] = M[j, i, k] % MOD
        R.append(Rmat)
    return R


def is_scalar(v):
    """Return scalar value if v = λ·1, else None."""
    for i in range(N):
        if i == 7:
            continue
        if v[i] != 0:
            return None
    return int(v[7])


def analyze(M, name, lengths=(2, 4, 6, 8)):
    print("=" * 72)
    print(f"LOOP ANALYSIS: {name}")
    print("=" * 72)
    print()

    L = left_matrices(M)
    R = right_matrices(M)

    # Precompute basis vectors
    basis = [np.zeros(N, dtype=int) for _ in range(N)]
    for i in range(N):
        basis[i][i] = 1

    for length in lengths:
        total = N ** length
        print(f"Length {length}  ({total:,} sequences)")

        # Left-associative: v = L[a_L] @ ... @ L[a_2] @ e_{a_1}
        # We iterate as: v = e_{a_1}; then v = L[a_2] @ v; ... ; v = L[a_L] @ v
        # Equivalently, fold left with L.
        cnt_L = 0
        sum_L = 0

        # Right-associative: v = R[a_1] @ ... @ R[a_{L-1}] @ e_{a_L}
        cnt_R = 0
        sum_R = 0

        # Progress
        report_every = max(1, total // 4)

        for idx, seq in enumerate(product(range(N), repeat=length)):
            # Left-assoc
            v = basis[seq[0]].copy()
            for a in seq[1:]:
                v = (L[a] @ v) % MOD
            s = is_scalar(v)
            if s is not None:
                cnt_L += 1
                sum_L += s

            # Right-assoc
            v = basis[seq[-1]].copy()
            for a in reversed(seq[:-1]):
                v = (R[a] @ v) % MOD
            s = is_scalar(v)
            if s is not None:
                cnt_R += 1
                sum_R += s

            if (idx + 1) % report_every == 0:
                pct = 100 * (idx + 1) / total
                print(f"  progress: {pct:.0f}%")

        print(f"  Left-assoc:  loops = {cnt_L:>10d}, "
              f"sum = {sum_L:>14d}, avg = {sum_L / max(1, cnt_L):>12.6f}")
        print(f"  Right-assoc: loops = {cnt_R:>10d}, "
              f"sum = {sum_R:>14d}, avg = {sum_R / max(1, cnt_R):>12.6f}")
        print()


M_E = build(t_sq=0, t_left=3, t_right=-3)
M_G = build(t_sq=-1, t_left=1, t_right=-1)

print("Starting Gap 11 loop analysis...")
print()

# Length 2, 4 for E; then 6, 8
analyze(M_E, "E (drain)", lengths=(2, 4))
analyze(M_G, "G (oscillator)", lengths=(2, 4))

# Length 6, 8 — this will be slower
print("=" * 72)
print("Proceeding to length 6 and 8 (may take a few minutes)...")
print("=" * 72)
print()

analyze(M_E, "E (drain) — length 6, 8", lengths=(6, 8))
analyze(M_G, "G (oscillator) — length 6, 8", lengths=(6, 8))

Starting Gap 11 loop analysis...

LOOP ANALYSIS: E (drain)

Length 2  (81 sequences)
  progress: 25%
  progress: 49%
  progress: 74%
  progress: 99%
  Left-assoc:  loops =          9, sum =             57, avg =     6.333333
  Right-assoc: loops =          9, sum =             57, avg =     6.333333

Length 4  (6,561 sequences)
  progress: 25%
  progress: 50%
  progress: 75%
  progress: 100%
  Left-assoc:  loops =       1199, sum =           3074, avg =     2.563803
  Right-assoc: loops =       1101, sum =           3074, avg =     2.792007

LOOP ANALYSIS: G (oscillator)

Length 2  (81 sequences)
  progress: 25%
  progress: 49%
  progress: 74%
  progress: 99%
  Left-assoc:  loops =          9, sum =             65, avg =     7.222222
  Right-assoc: loops =          9, sum =             65, avg =     7.222222

Length 4  (6,561 sequences)
  progress: 25%
  progress: 50%
  progress: 75%
  progress: 100%
  Left-assoc:  loops =        845, sum =           3190, avg =     3.775148
  Right-as